In [ ]:
import torch
import torch.nn as nn

class TextClassifierMLP(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int, num_classes: int, dropout: float = 0.3):
        super().__init__()

        # входной слой
        self.embedding = nn.EmbeddingBag(vocab_size, embed_dim, mode='mean')

        # скрытый слой
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.act1 = nn.GELU() # новая альтернатива ReLU
        self.dropout1 = nn.Dropout(dropout) # отключает часть нейронов в эпохе

        # выходной слой (логиты для каждого класса)
        self.fc2 = nn.Linear(hidden_dim, num_classes)

        # явная инилизация весов
        self._init_weights()

    def _init_weights(self):
        """инилизация Kaiming (He) для линейных слоев с GELU/Relu"""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.0)
                    
    def forward(self, text: torch.Tensor, offsets: torch.Tensor) -> torch.Tensor:
        x = self.embedding(text, offsets)
        x = self.fc1(x)
        x = self.act1(x)
        x = self.dropout1(x)
        logits = self.fc2(x)
        return logits
    
                